# 80 — Global Roman allocation and closure checkpoint

This workbook joins the exact ledger to the closure-aware Constantine
card and the reduced neutron-transport scenarios. It does not optimize
fusion gain. It asks whether one net N15 burn, explicit DT/DD support,
one desired Constantine neutron, and recovered catalysts can coexist
at one completed traversal of every recipe.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
while repo_root.name != 'CNO' and repo_root != repo_root.parent:
    repo_root = repo_root.parent
source_root = repo_root / 'analysis' / 'src'
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
result_root = repo_root / 'analysis' / 'results' / 'roman-workbooks'
result_root.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.precision', 6)

from math import exp
import cno_sweep.neutron_transport as nt
from cno_sweep import (
    Material, driver_support_from_targets, equal_success_probability_floor,
    evaluate_roman_closure, expanding_core_release,
    load_builtin_neutron_cross_sections, number_densities,
    repeated_blanket_capture_probability, roman_phase_one_closure_target_sets,
)

xs = load_builtin_neutron_cross_sections()
targets = roman_phase_one_closure_target_sets()
supports = {case: driver_support_from_targets(case, cards) for case, cards in targets.items()}

In [2]:
support_rows = []
for case_name, support in supports.items():
    support_rows.append({
        'case': case_name,
        'calculated N15 burn': support.calculated_n15_burn_per_traversal,
        'unallocated N15 burn reserve': support.n15_burn_reserve_per_traversal,
        'N15 loaded': support.n15_loaded_for_exact_burn,
        'N15 recovered unburned': support.n15_unburned_after_exact_burn,
        'driver DT loaded': support.driver_dt_loaded_for_exact_n15_burn,
        'driver DT burned': support.driver_dt_burned_for_exact_n15_burn,
        'central DT loaded': support.central_dt_loaded_per_traversal,
        'central DT burned': support.central_dt_burned_per_traversal,
        'all DT burned P': support.total_dt_burned_per_traversal,
        'equal-shot success floor if failed driver fully burns': equal_success_probability_floor(support),
    })
display(pd.DataFrame(support_rows).set_index('case'))

,calculated N15 burn,unallocated N15 burn reserve,N15 loaded,N15 recovered unburned,driver DT loaded,driver DT burned,central DT loaded,central DT burned,all DT burned P,equal-shot success floor if failed driver fully burns
case,,,,,,,,,,
likely,0.996965,0.003035,2.000000,1.000000,0.111200,0.088960,0.052897,0.026448,0.115408,0.996965
conservative,0.959266,0.040734,2.857143,1.857143,0.228571,0.137143,0.000049,0.000015,0.137157,0.959266


Exactly one N15 is burned per traversal. `calculated N15 burn` is the
amount needed by the five selected driver cards; the remainder is
overdrive/retry reserve within that same one atom, not extra fuel.
Loaded N15 exceeds one only because unburned enriched nitrogen is
recovered. Tritium replacement is explicit: every lost/burned triton
requires one DD tritium branch plus its statistically companion DD
neutron branch, consuming four D in addition to the D in DT.

The scenario table below uses perfect recovery of unburned target
inventory so the neutron question stays isolated. Workbook 10 contains
the recovery-loss sweep. At 90% recovery of unburned D/T, both stated
working points remain D-positive, but lost N15 and heavy catalyst become
explicit makeup feeds rather than a closed internal inventory.

In [3]:
allocation_rows = []
for case_name, support in supports.items():
    for recipe, allocation in support.n15_allocations.items():
        allocation_rows.append({
            'case': case_name, 'recipe': recipe,
            'N15 burned as driver / traversal': allocation,
            'core heavy burn fraction': support.core_burn_fractions[recipe],
        })
display(pd.DataFrame(allocation_rows).pivot(
    index='recipe', columns='case', values='N15 burned as driver / traversal'
))

case,conservative,likely
recipe,,
aurelian,0.455698,0.262672
caesar,0.022012,0.034569
constantine,0.013137,0.013327
diocletian,0.450995,0.669339
scipio,0.017425,0.017058


In [4]:
h2 = Material('ordinary H2', number_densities(70.8, {'h1': 1.0}, xs), True)
capture_one_entry = 1.0 - exp(
    -nt.transport_mean_free_path_m(h2, xs) / nt.diffusion_length_m(h2, xs)
)

scenario_specs = {
    'likely working point': {
        'case': 'likely', 'expansion_multiplier': 3.0,
        'target_return_survival': 0.98, 'eta_dt': 0.75, 'eta_dd': 0.95,
    },
    'conservative working point': {
        'case': 'conservative', 'expansion_multiplier': 10.0,
        'target_return_survival': 0.95, 'eta_dt': 0.55, 'eta_dd': 0.90,
    },
    'conservative stress point': {
        'case': 'conservative', 'expansion_multiplier': 30.0,
        'target_return_survival': 0.90, 'eta_dt': 0.35, 'eta_dd': 0.80,
    },
}
global_rows = []
for scenario_name, spec in scenario_specs.items():
    case_name = spec['case']
    state = targets[case_name]['constantine'].radius_state
    core = Material('C13/alpha', number_densities(
        state.compressed_density_kg_m3, {'c13': 1.0, 'he4': 1.0}, xs
    ))
    release = expanding_core_release(
        core, xs, source_energy_mev=2.2,
        ion_temperature_keV=state.ion_temperature_keV,
        radius_m=state.compressed_fuel_radius_m,
        hydrodynamic_time_s=state.hydrodynamic_time_s,
        expansion_time_multiplier=spec['expansion_multiplier'],
    ).release_probability
    blanket = repeated_blanket_capture_probability(
        capture_one_entry, spec['target_return_survival']
    )
    eta_c = release * blanket
    ledger = evaluate_roman_closure(
        supports[case_name], eta_constantine_n_to_d=eta_c,
        eta_dt_n_to_d=spec['eta_dt'], eta_dd_n_to_d=spec['eta_dd'],
    )
    global_rows.append({
        'scenario': scenario_name, **spec,
        'eta Constantine': eta_c,
        'DT loaded': ledger.dt_loaded,
        'DT burned': ledger.dt_burned,
        'DD tritium reactions': ledger.dd_tritium_branch_reactions,
        'DD neutron reactions': ledger.dd_neutron_branch_reactions,
        'D total consumed': ledger.d_total_consumed,
        'D from Constantine n': ledger.d_recovered_from_constantine,
        'D from DT n': ledger.d_recovered_from_dt_neutrons,
        'D from DD n': ledger.d_recovered_from_dd_neutrons,
        'D total recovered': ledger.d_total_recovered,
        'net D': ledger.delta_d,
        'G_D': ledger.g_d,
        'net alpha': ledger.total_alpha_surplus,
        'net T': ledger.t_net,
    })
global_table = pd.DataFrame(global_rows).set_index('scenario')
display(global_table)

,case,expansion_multiplier,target_return_survival,eta_dt,eta_dd,eta Constantine,DT loaded,DT burned,DD tritium reactions,DD neutron reactions,D total consumed,D from Constantine n,D from DT n,D from DD n,D total recovered,net D,G_D,net alpha,net T
scenario,,,,,,,,,,,,,,,,,,,
likely working point,likely,3.0,0.98,0.75,0.95,0.849117,0.164097,0.115408,0.115408,0.115408,0.577042,0.849117,0.086556,0.109638,1.045311,0.468270,1.811501,1.115408,0.0
conservative working point,conservative,10.0,0.95,0.55,0.90,0.602623,0.228620,0.137157,0.137157,0.137157,0.685787,0.602623,0.075437,0.123442,0.801502,0.115714,1.168732,1.137157,0.0
conservative stress point,conservative,30.0,0.90,0.35,0.80,0.344324,0.228620,0.137157,0.137157,0.137157,0.685787,0.344324,0.048005,0.109726,0.502055,-0.183732,0.732086,1.137157,0.0


In [5]:
# Exact conservation-oriented summary for the two working rows.
flow_rows = []
for row in global_rows[:2]:
    flow_rows.extend([
        {'scenario': row['scenario'], 'flow': 'mainline Constantine neutrons', 'amount': 1.0},
        {'scenario': row['scenario'], 'flow': 'DT reactions / neutrons / alphas', 'amount': row['DT burned']},
        {'scenario': row['scenario'], 'flow': 'DD -> T branch reactions', 'amount': row['DD tritium reactions']},
        {'scenario': row['scenario'], 'flow': 'DD -> n branch reactions', 'amount': row['DD neutron reactions']},
        {'scenario': row['scenario'], 'flow': 'D consumed', 'amount': row['D total consumed']},
        {'scenario': row['scenario'], 'flow': 'recoverable D produced', 'amount': row['D total recovered']},
        {'scenario': row['scenario'], 'flow': 'net D', 'amount': row['net D']},
        {'scenario': row['scenario'], 'flow': 'net T', 'amount': row['net T']},
        {'scenario': row['scenario'], 'flow': 'net alpha including DT', 'amount': row['net alpha']},
    ])
display(pd.DataFrame(flow_rows).pivot(index='flow', columns='scenario', values='amount'))

scenario,conservative working point,likely working point
flow,,
D consumed,0.685787,0.577042
DD -> T branch reactions,0.137157,0.115408
DD -> n branch reactions,0.137157,0.115408
DT reactions / neutrons / alphas,0.137157,0.115408
mainline Constantine neutrons,1.000000,1.000000
net D,0.115714,0.468270
net T,0.000000,0.000000
net alpha including DT,1.137157,1.115408
recoverable D produced,0.801502,1.045311


## Checkpoint judgment

The algebraic cycle is closed exactly: the heavy catalyst returns to
C12, one N15 returns through Trajan, the alpha borrowed by Constantine
is repaid and one additional mainline alpha remains, and DD support
replaces every triton used by DT.

The reduced physical model also has D-positive likely and conservative
working points. The conservative stress point is intentionally not
closed, which identifies the remaining gate: neutron release and
repeated target/H-blanket albedo must be verified on the simulated
disassembly history. In other words, closure is now demonstrated as a
physically parameterized possibility, not assumed unconditionally.

In [6]:
artifact = {
    'schema': 'roman-global-allocation-v0.1',
    'normalization': 'one completed heavy-catalyst traversal',
    'support_rows': support_rows,
    'allocation_rows': allocation_rows,
    'scenario_rows': global_rows,
}
path = result_root / '80-global-allocation-v0.1.json'
path.write_text(json.dumps(artifact, indent=2))
print('wrote', path.relative_to(repo_root))

wrote analysis/results/roman-workbooks/80-global-allocation-v0.1.json
